In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Cell 1: Library dasar siap!")

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Ambil ID untuk submission
test_ids = test_df['PassengerId']
print("Cell 2: Data Train dan Test berhasil dimuat.")

In [ ]:
full_df = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)

# 1. Ekstrak Gelar
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master'}
full_df['Title'] = full_df['Title'].map(title_mapping).fillna('Rare')

# 2. Ukuran Keluarga
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1

# 3. Kembalikan Fitur Deck dari Cabin (Sinyal Kelas Sosial)
full_df['Deck'] = full_df['Cabin'].str[0].fillna('U')

# 4. Imputasi Anti-NaN
full_df['Age'] = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
full_df['Age'] = full_df['Age'].fillna(full_df['Age'].median())

full_df['Fare'] = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full_df['Fare'] = full_df['Fare'].fillna(full_df['Fare'].median())
full_df['Embarked'] = full_df['Embarked'].fillna('S')

# 5. Binning (Mengelompokkan Umur & Tarif menjadi 5 kategori untuk membuang noise)
full_df['Age_Bin'] = pd.qcut(full_df['Age'], 5, labels=[0, 1, 2, 3, 4]).astype(int)
full_df['Fare_Bin'] = pd.qcut(full_df['Fare'], 5, labels=[0, 1, 2, 3, 4]).astype(int)

print("Cell 3: Pembersihan dan penghilangan noise berhasil!")

In [ ]:
full_df['Surname'] = full_df['Name'].apply(lambda x: x.split(',')[0].strip())
full_df['Group_Survival'] = 0.5 # Default (Tidak diketahui)

# Lacak berdasarkan Nama Belakang dan Tarif (Keluarga)
for grp, grp_df in full_df.groupby(['Surname', 'Fare']):
    if len(grp_df) > 1:
        for ind, row in grp_df.iterrows():
            smax = grp_df.drop(ind)['Survived'].max()
            smin = grp_df.drop(ind)['Survived'].min()
            passID = row['PassengerId']
            if smax == 1.0:
                full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 1
            elif smin == 0.0:
                full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 0

# Lacak berdasarkan Tiket (Rombongan/Pengasuh/Teman)
for grp, grp_df in full_df.groupby('Ticket'):
    if len(grp_df) > 1:
        for ind, row in grp_df.iterrows():
            if full_df.loc[full_df['PassengerId'] == row['PassengerId'], 'Group_Survival'].values[0] == 0.5:
                smax = grp_df.drop(ind)['Survived'].max()
                smin = grp_df.drop(ind)['Survived'].min()
                passID = row['PassengerId']
                if smax == 1.0:
                    full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 1
                elif smin == 0.0:
                    full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 0

# Simpan riwayat ini ke variabel terpisah untuk tahap override nanti
group_survival_history = full_df['Group_Survival'].copy()

print("Cell 4: Riwayat keselamatan keluarga/rombongan direkam!")

In [ ]:
# Hapus kolom mentah, TERMASUK Age, Fare (diganti Age_Bin/Fare_Bin) dan Group_Survival (digunakan manual nanti)
drop_cols = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin', 'Surname', 'Age', 'Fare', 'Group_Survival']

# One-Hot Encoding
categorical_cols = ['Sex', 'Embarked', 'Title', 'Deck']
full_df = pd.get_dummies(full_df, columns=categorical_cols, drop_first=True, dtype=int)

# Pemisahan Train dan Test
train_clean = full_df[full_df['Survived'].notnull()].copy()
test_clean = full_df[full_df['Survived'].isnull()].copy()

X = train_clean.drop(columns=drop_cols, errors='ignore')
y = train_clean['Survived'].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# Penyelarasan
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"Cell 5: Matriks siap. Jumlah fitur: {X.shape[1]}")

In [ ]:
# Kita kembali gunakan Random Forest, namun diberi batasan ketat (Regularisasi)
# min_samples_leaf=2 mencegah model membuat cabang untuk data tunggal (mencegah hafalan)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=2, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')

print(f"Akurasi ML Murni (Cross-Validation): {scores.mean() * 100:.2f}%")

rf_model.fit(X, y)
print("Cell 6: Model berhasil dilatih!")

In [ ]:
# 1. Biarkan model menebak nasib seluruh penumpang ujian
ml_predictions = rf_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': ml_predictions
})

# 2. Ekstrak data Group_Survival khusus untuk penumpang ujian (test_df)
test_group_survival = group_survival_history[full_df['Survived'].isnull()].values

# 3. Eksekusi RULE-BASED OVERRIDE (Timpakan fakta historis ke atas tebakan ML)
overridden_count = 0
for i in range(len(submission)):
    if test_group_survival[i] == 1.0:       # Jika terbukti ada keluarganya yang selamat
        submission.loc[i, 'Survived'] = 1
        overridden_count += 1
    elif test_group_survival[i] == 0.0:     # Jika terbukti seluruh keluarganya gugur
        submission.loc[i, 'Survived'] = 0
        overridden_count += 1

# 4. Ekspor ke CSV
submission.to_csv('submission_titanic_hybrid_rulebased.csv', index=False)

print(f"Cell 7: File 'submission_titanic_hybrid_rulebased.csv' berhasil dibuat!")
print(f"Terdapat {overridden_count} tebakan model yang ditimpa secara paksa oleh aturan historis rombongan.")